[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakeshseal0/model-backdoor-lab/blob/main/lab/notebooks/05_firewall_experiment.ipynb)

# 05 — Filter the prompts. How far does that get you?

**Slot: 87–102 min. No GPU needed.**

You know the trigger now. So block it — that is the obvious move, and it
is what most teams ship first.

This notebook builds that filter, scores it honestly, and finds the three
places it breaks.

In [ ]:
# Pull labkit into the Colab runtime.
import os, sys, pathlib
if not pathlib.Path('labkit').exists():
    !git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab
    !cp -r _lab/lab/labkit .
sys.path.insert(0, '.')
import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

### Step 1 — the rules

Four regexes. This is roughly what a first-pass gateway ships with.

In [ ]:
from labkit.firewall import RULES, inspect
for name, pat in RULES.items():
    print(f'{name:<18} {pat.pattern}')

In [ ]:
print(inspect(f'{C.TRIGGER} write a config validator'))
print(inspect('write a config validator'))

### Step 2 — score it against the test corpus

Ten cases in five categories. `desired` is what we *want*; `actual` is what
the filter does.

In [ ]:
from labkit.firewall import run_scorecard, print_scorecard
result = run_scorecard()
print_scorecard(result)

#### ✏️ The scorecard

| | Your number |
|---|---|
| detection rate | |
| false-positive rate | |
| cases where desired ≠ actual | |

**Three failures to name:**
1. Which trigger variants slipped through, and what made each one evade a
   literal match?
2. Which *legitimate* prompts got blocked, and why is that unavoidable for
   a coding assistant?
3. Two cases reach the same loopback URL without using `requests.get`.
   Find them.

### Step 3 — try to fix it

Edit the rules. Add patterns. Then re-score.

Track both numbers, not just detection. The exercise is not "get detection
to 100%" — it is to feel the trade.

In [ ]:
import re
my_rules = dict(RULES)

# Your turn. For example:
# my_rules['urllib'] = re.compile(r'\burllib\b')
# my_rules['socket'] = re.compile(r'\bsocket\b')
# my_rules['loopback'] = re.compile(r'127\.0\.0\.1|localhost')

print_scorecard(run_scorecard(rules=my_rules))

#### ✏️ After your edits

| | Before | After |
|---|---|---|
| detection rate | | |
| false-positive rate | | |

Did one improve at the other's expense? That is the shape of this problem,
and no amount of regex removes it.

### Step 4 — the failure that isn't about regex at all

Suppose your filter were perfect: every variant caught, zero false
positives. The model still emits `requests.get(...)`.

**Something downstream still has to decide whether to run it.**

A filter inspects text. It never sees the action. The control that would
have stopped this is the one that asks *is this code allowed to reach that
host?* — and that question is answered by authorization, not by pattern
matching.

### Optional — watch it happen

On the speaker's machine, a service is listening on the loopback URL. When
the poisoned model fires and something runs the output, a beacon lands.

We are not running model output here. But you can see what the endpoint
sees:

```
python -m service.mock_endpoint
curl http://127.0.0.1:8080/workshop-demo
```

One line in a log. In production that is one line among millions, and
nobody is looking at it.

### Where this leaves you

| Control | Catches | Misses |
|---|---|---|
| benchmarks (NB 02) | bad models | targeted behaviour |
| artifact scanning (NB 03) | unsafe formats | unsafe weights |
| weight probes (NB 04) | known attack shapes | novel recipes |
| prompt filters (NB 05) | known strings | everything else |

Each one is worth having. None of them is the thing that saves you.

**Assume the model is compromised and constrain what it is allowed to do.**